# Text Vectorization Assignment

One-Hot, Bag of Words, N-grams, TF-IDF  
(No ML model training — feature extraction only)

**Dataset:** Same 25 product-review samples from previous NLP assignment.  
Using cleaned text (`final_clean_text` style).


## Setup & Data


In [1]:
import re
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# cleaned review texts (from previous preprocessing style)
texts = [
    "phone amazing battery lasts days buy now",
    "worst laptop ever screen broke weeks contact help",
    "great headphones sound quality superb price only",
    "love camera photos sharp visit more reviews",
    "shoes comfortable size small ordered pairs yesterday",
    "buy fake product email need proof",
    "nice watch delivery fast tracking",
    "book okay pages missing rating stars",
    "amazing blender makes smoothies seconds highly recommend",
    "tablet freezes often updated software times still issue",
    "bag looks premium material quality good price rs",
    "keyboard keys sticky returned march",
    "fantastic service seller got refund within hours thanks",
    "perfume smell lasts hour worth rupees",
    "mouse wireless smooth battery life months",
    "shirt fabric soft color faded washes",
    "check full review item gadget",
    "chair ergonomic back pain reduced week use",
    "power bank capacity less advertised shown",
    "sunglasses look stylish uv protection works well outdoors",
    "app crashes android contact",
    "coffee maker easy clean makes cups minutes",
    "speaker bass strong connectivity drops sometimes",
    "notebook quality average pages thin price",
    "product good great quality buy"
]

df = pd.DataFrame({'id': range(1, 26), 'final_clean_text': texts})
print('Samples:', len(df))
print(df.head())


Samples: 25
   id                                   final_clean_text
0   1           phone amazing battery lasts days buy now
1   2  worst laptop ever screen broke weeks contact help
2   3   great headphones sound quality superb price only
3   4        love camera photos sharp visit more reviews
4   5  shoes comfortable size small ordered pairs yes...


## PART 1 — One-Hot Encoding (Text Level)

### Task 1: Manual One-Hot Encoding


In [2]:
# select 5 short sentences
sample = df['final_clean_text'].head(5).tolist()
print('Selected sentences:')
for i, s in enumerate(sample):
    print(i, s)

# build vocabulary
vocab = sorted(set(' '.join(sample).split()))
print('\nVocabulary:', vocab)
print('Vocab size:', len(vocab))

# one-hot vectors manually
def one_hot(sentence, vocab):
    words = set(sentence.split())
    return [1 if w in words else 0 for w in vocab]

matrix = [one_hot(s, vocab) for s in sample]
oh_df = pd.DataFrame(matrix, columns=vocab)
print('\nOne-hot matrix:')
print(oh_df)


Selected sentences:
0 phone amazing battery lasts days buy now
1 worst laptop ever screen broke weeks contact help
2 great headphones sound quality superb price only
3 love camera photos sharp visit more reviews
4 shoes comfortable size small ordered pairs yesterday

Vocabulary: ['amazing', 'battery', 'broke', 'buy', 'camera', 'comfortable', 'contact', 'days', 'ever', 'great', 'headphones', 'help', 'laptop', 'lasts', 'love', 'more', 'now', 'only', 'ordered', 'pairs', 'phone', 'photos', 'price', 'quality', 'reviews', 'screen', 'sharp', 'shoes', 'size', 'small', 'sound', 'superb', 'visit', 'weeks', 'worst', 'yesterday']
Vocab size: 36

One-hot matrix:
   amazing  battery  broke  buy  camera  comfortable  contact  days  ever  \
0        1        1      0    1       0            0        0     1     0   
1        0        0      1    0       0            0        1     0     1   
2        0        0      0    0       0            0        0     0     0   
3        0        0      0    0   

### Task 2: One-Hot Encoding Using Scikit-learn


In [3]:
# binary CountVectorizer = one-hot style
cv_bin = CountVectorizer(binary=True)
X_bin = cv_bin.fit_transform(sample)

print('Vocabulary:', cv_bin.get_feature_names_out().tolist())
print('\nEncoded matrix:')
print(pd.DataFrame(X_bin.toarray(), columns=cv_bin.get_feature_names_out()))


Vocabulary: ['amazing', 'battery', 'broke', 'buy', 'camera', 'comfortable', 'contact', 'days', 'ever', 'great', 'headphones', 'help', 'laptop', 'lasts', 'love', 'more', 'now', 'only', 'ordered', 'pairs', 'phone', 'photos', 'price', 'quality', 'reviews', 'screen', 'sharp', 'shoes', 'size', 'small', 'sound', 'superb', 'visit', 'weeks', 'worst', 'yesterday']

Encoded matrix:
   amazing  battery  broke  buy  camera  comfortable  contact  days  ever  \
0        1        1      0    1       0            0        0     1     0   
1        0        0      1    0       0            0        1     0     1   
2        0        0      0    0       0            0        0     0     0   
3        0        0      0    0       1            0        0     0     0   
4        0        0      0    0       0            1        0     0     0   

   great  ...  sharp  shoes  size  small  sound  superb  visit  weeks  worst  \
0      0  ...      0      0     0      0      0       0      0      0      0   
1 

## PART 2 — Bag of Words (BoW)

### Task 3: Bag of Words Representation


In [4]:
cv = CountVectorizer()
X_bow = cv.fit_transform(df['final_clean_text'])

print('Vocabulary size:', len(cv.get_feature_names_out()))
print('Sample feature vectors (first 3 docs):')
print(pd.DataFrame(X_bow.toarray()[:3], columns=cv.get_feature_names_out()).iloc[:, :12])


Vocabulary size: 145
Sample feature vectors (first 3 docs):
   advertised  amazing  android  app  average  back  bag  bank  bass  battery  \
0           0        1        0    0        0     0    0     0     0        1   
1           0        0        0    0        0     0    0     0     0        0   
2           0        0        0    0        0     0    0     0     0        0   

   blender  book  
0        0     0  
1        0     0  
2        0     0  


### Task 4: Understanding Word Frequency


In [5]:
# total frequency of each word
freqs = X_bow.toarray().sum(axis=0)
word_freq = pd.Series(freqs, index=cv.get_feature_names_out()).sort_values(ascending=False)

print('Top 10 most frequent words:')
print(word_freq.head(10))
print('\nLeast frequent words:')
print(word_freq.tail(10))

print('''
BoW captures frequency by counting how many times each word appears
in a document. Higher count = stronger presence of that word.
''')


Top 10 most frequent words:
quality    4
price      3
buy        3
battery    2
amazing    2
great      2
contact    2
pages      2
makes      2
lasts      2
dtype: int64

Least frequent words:
watch        1
week         1
weeks        1
well         1
wireless     1
within       1
works        1
worst        1
worth        1
yesterday    1
dtype: int64

BoW captures frequency by counting how many times each word appears
in a document. Higher count = stronger presence of that word.



## PART 3 — N-Grams

### Task 5: Unigrams, Bigrams & Trigrams


In [6]:
for n, name in [(1, 'Unigrams'), (2, 'Bigrams'), (3, 'Trigrams')]:
    v = CountVectorizer(ngram_range=(n, n))
    X = v.fit_transform(df['final_clean_text'])
    print(f'{name}: vocab size = {len(v.get_feature_names_out())}')
    print('Sample features:', v.get_feature_names_out()[:8].tolist())
    print()


Unigrams: vocab size = 145
Sample features: ['advertised', 'amazing', 'android', 'app', 'average', 'back', 'bag', 'bank']

Bigrams: vocab size = 136
Sample features: ['advertised shown', 'amazing battery', 'amazing blender', 'android contact', 'app crashes', 'average pages', 'back pain', 'bag looks']

Trigrams: vocab size = 111
Sample features: ['amazing battery lasts', 'amazing blender makes', 'app crashes android', 'average pages thin', 'back pain reduced', 'bag looks premium', 'bank capacity less', 'bass strong connectivity']



### Task 6: Combined N-Grams


In [7]:
v12 = CountVectorizer(ngram_range=(1, 2))
X12 = v12.fit_transform(df['final_clean_text'])
print('Unigram+Bigram vocab size:', len(v12.get_feature_names_out()))
print('Sample features:', v12.get_feature_names_out()[:15].tolist())
print('''
Using (1,2) keeps single words and adds 2-word phrases.
This adds context (e.g. "battery life") compared to unigrams alone.
''')


Unigram+Bigram vocab size: 281
Sample features: ['advertised', 'advertised shown', 'amazing', 'amazing battery', 'amazing blender', 'android', 'android contact', 'app', 'app crashes', 'average', 'average pages', 'back', 'back pain', 'bag', 'bag looks']

Using (1,2) keeps single words and adds 2-word phrases.
This adds context (e.g. "battery life") compared to unigrams alone.



## PART 4 — TF-IDF Vectorization

### Task 7: TF-IDF Implementation


In [8]:
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(df['final_clean_text'])

print('Vocabulary size:', len(tfidf.get_feature_names_out()))
print('TF-IDF matrix shape:', X_tfidf.shape)
print('\nSample TF-IDF values (first 2 docs, first 10 features):')
print(pd.DataFrame(X_tfidf.toarray()[:2], columns=tfidf.get_feature_names_out()).iloc[:, :10].round(3))


Vocabulary size: 145
TF-IDF matrix shape: (25, 145)

Sample TF-IDF values (first 2 docs, first 10 features):
   advertised  amazing  android  app  average  back  bag  bank  bass  battery
0         0.0    0.362      0.0  0.0      0.0   0.0  0.0   0.0   0.0    0.362
1         0.0    0.000      0.0  0.0      0.0   0.0  0.0   0.0   0.0    0.000


### Task 8: BoW vs TF-IDF Comparison


In [9]:
# average tf-idf weight per word
avg_tfidf = X_tfidf.toarray().mean(axis=0)
tfidf_weights = pd.Series(avg_tfidf, index=tfidf.get_feature_names_out()).sort_values(ascending=False)

print('Words with high TF-IDF (more distinctive):')
print(tfidf_weights.head(10).round(4))
print('\nWords with low TF-IDF (more common):')
print(tfidf_weights.tail(10).round(4))

print('''
TF-IDF down-weights common words because they appear in many documents
(low IDF). Rare but meaningful words get higher weight.
''')


Words with high TF-IDF (more distinctive):
quality    0.0525
buy        0.0441
price      0.0398
product    0.0340
great      0.0333
good       0.0322
contact    0.0309
pages      0.0306
battery    0.0292
lasts      0.0292
dtype: float64

Words with low TF-IDF (more common):
software      0.0141
sunglasses    0.0141
times         0.0141
thanks        0.0141
tablet        0.0141
uv            0.0141
updated       0.0141
well          0.0141
within        0.0141
works         0.0141
dtype: float64

TF-IDF down-weights common words because they appear in many documents
(low IDF). Rare but meaningful words get higher weight.



## PART 5 — Practical Insights

### Task 9: Vectorizer Parameter Exploration


In [10]:
print('Default vocab size:', len(CountVectorizer().fit(df['final_clean_text']).get_feature_names_out()))

for max_f in [20, 50]:
    v = CountVectorizer(max_features=max_f)
    v.fit(df['final_clean_text'])
    print(f'max_features={max_f} -> vocab size={len(v.get_feature_names_out())}')

for min_df in [2, 3]:
    v = CountVectorizer(min_df=min_df)
    v.fit(df['final_clean_text'])
    print(f'min_df={min_df} -> vocab size={len(v.get_feature_names_out())}')

for max_df in [0.8, 0.5]:
    v = CountVectorizer(max_df=max_df)
    v.fit(df['final_clean_text'])
    print(f'max_df={max_df} -> vocab size={len(v.get_feature_names_out())}')


Default vocab size: 145
max_features=20 -> vocab size=20
max_features=50 -> vocab size=50
min_df=2 -> vocab size=12
min_df=3 -> vocab size=3
max_df=0.8 -> vocab size=145
max_df=0.5 -> vocab size=145


### Task 10: Conceptual Questions


In [11]:
print('''
1. Difference between One-Hot Encoding and BoW:
   One-hot (binary) only marks presence (0/1).
   BoW counts how many times each word appears.

2. Why N-grams increase dimensionality:
   Each unique pair/triple becomes a new feature.
   More combinations => much larger vocabulary.

3. When to prefer TF-IDF over BoW:
   When common words should matter less and rare informative
   words should matter more (most text classification tasks).

4. Limitations of count-based vectorization:
   - Ignores word order (except with n-grams)
   - Large sparse matrices
   - No real meaning/semantics (synonyms treated as different)
''')



1. Difference between One-Hot Encoding and BoW:
   One-hot (binary) only marks presence (0/1).
   BoW counts how many times each word appears.

2. Why N-grams increase dimensionality:
   Each unique pair/triple becomes a new feature.
   More combinations => much larger vocabulary.

3. When to prefer TF-IDF over BoW:
   When common words should matter less and rare informative
   words should matter more (most text classification tasks).

4. Limitations of count-based vectorization:
   - Ignores word order (except with n-grams)
   - Large sparse matrices
   - No real meaning/semantics (synonyms treated as different)

